# V20 – Datenbanken Teil 2: Python-Recap

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- mit `dict` und `list[dict]` Datensätze im Gedächtnis halten,
- mit `sorted(..., key=lambda ...)` und `max`/`min` mit `key` nach beliebigen Feldern sortieren,
- eine **List-Comprehension** einsetzen, um Zeilen zu filtern oder umzuformen,
- dich an die **sqlite3-Grundlagen** aus V19 erinnern (`connect(":memory:")`, `cursor.execute`, `fetchall`).

## ⏱️ Zeitbudget
Ca. 15 Minuten.

## 🧭 TL;DR
Wir frischen nur die Python-Werkzeuge auf, die wir in V20 wirklich brauchen: Daten in Listen von Dictionaries halten, sortieren, filtern – und ganz am Ende drei Zeilen `sqlite3` als Wiederholung.

## 📦 Voraussetzungen
- V19 – Datenbanken Teil 1
- Grundlagen `dict`, `list`, `for` (V08)


## 1. `dict` als Datensatz

Ein **dict** (Dictionary) speichert Daten unter benannten Schlüsseln. Für tabellenähnliche Daten wie Zeilen aus einer Datenbank ist das die natürliche Darstellung: Jeder Spaltenname wird ein Schlüssel, jeder Zellwert der zugehörige Wert.

Ein Dictionary ersetzt damit fehleranfällige Positions-Tupel wie `(3, "Drehmaschine 01", 2500.0)`, bei denen man sich merken müsste, welche Position welche Bedeutung hat. Mit einem Dictionary schreibt man stattdessen `zeile["kosten"]` und liest den Code selbsterklärend.

In [1]:
maschine = {
    "id": 1,
    "name": "CNC-Fräse 01",
    "typ": "Fräse",
    "baujahr": 2018,
}
print(maschine["name"], "-", maschine["baujahr"])


CNC-Fräse 01 - 2018


## 2. `list[dict]` als Tabelle

Eine Liste von Dictionaries modelliert eine ganze Tabelle: Jede Liste ist eine Menge von Zeilen, jede Zeile ist ein Dictionary mit denselben Schlüsseln. Genau in dieser Form werden wir später Ergebnisse aus SQL-Abfragen weiterverarbeiten.

In [2]:
wartungen = [
    {"maschine": "CNC-Fräse 01",    "typ": "Inspektion", "kosten":  150.0},
    {"maschine": "CNC-Fräse 01",    "typ": "Reparatur",  "kosten": 1200.0},
    {"maschine": "Drehmaschine 01", "typ": "Reparatur",  "kosten": 2500.0},
    {"maschine": "Drehmaschine 01", "typ": "Ölwechsel",  "kosten":   90.0},
]
for w in wartungen:
    print(f"{w['maschine']:20s} {w['typ']:12s} {w['kosten']:>8.2f} €")


CNC-Fräse 01         Inspektion     150.00 €
CNC-Fräse 01         Reparatur     1200.00 €
Drehmaschine 01      Reparatur     2500.00 €
Drehmaschine 01      Ölwechsel       90.00 €


## 3. Sortieren mit `sorted` und `key=lambda`

Die eingebaute Funktion `sorted(iterable, key=..., reverse=...)` liefert eine **neue** sortierte Liste. Mit `key=lambda x: x[...]` geben wir an, **nach welchem Feld** sortiert werden soll. `reverse=True` dreht die Reihenfolge um, sodass das größte Element zuerst steht.

`lambda x: x["kosten"]` ist eine kleine, einzeilige Funktion: Sie nimmt `x` und liefert `x["kosten"]` zurück. Damit sagen wir `sorted`: „Sortiere jedes Element nach dem Wert seines Schlüssels `kosten`.

In [3]:
nach_kosten = sorted(wartungen, key=lambda w: w["kosten"], reverse=True)
for w in nach_kosten:
    print(f"{w['kosten']:>8.2f} € – {w['maschine']} ({w['typ']})")


 2500.00 € – Drehmaschine 01 (Reparatur)
 1200.00 € – CNC-Fräse 01 (Reparatur)
  150.00 € – CNC-Fräse 01 (Inspektion)
   90.00 € – Drehmaschine 01 (Ölwechsel)


## 4. `max` und `min` mit `key`

Wenn du nur **ein** Extremum brauchst, nicht die ganze sortierte Liste, sind `max(...)` und `min(...)` mit `key` das richtige Werkzeug. Sie laufen einmal über die Daten und geben das Element mit dem größten bzw. kleinsten Funktionswert zurück.

In [4]:
teuerste = max(wartungen, key=lambda w: w["kosten"])
guenstigste = min(wartungen, key=lambda w: w["kosten"])
print("Teuerste:   ", teuerste)
print("Günstigste: ", guenstigste)


Teuerste:    {'maschine': 'Drehmaschine 01', 'typ': 'Reparatur', 'kosten': 2500.0}
Günstigste:  {'maschine': 'Drehmaschine 01', 'typ': 'Ölwechsel', 'kosten': 90.0}


## 5. List-Comprehensions zum Filtern

Eine **List-Comprehension** der Form `[x for x in liste if bedingung]` erzeugt eine neue Liste aus allen Elementen, die die Bedingung erfüllen. Sie ist die kompakte Variante einer `for`-Schleife mit `if`, und wir brauchen sie, um Ergebnisse aus SQL-Abfragen nachträglich zu formen.

In [5]:
teure = [w for w in wartungen if w["kosten"] > 500]
namen = [w["maschine"] for w in teure]
print("Teure Wartungen (>500 €):", namen)


Teure Wartungen (>500 €): ['CNC-Fräse 01', 'Drehmaschine 01']


## 6. Kurzer `sqlite3`-Refresher (aus V19)

Die drei Zeilen, die wir ab sofort in jedem Notebook sehen werden, sind:

1. **Verbindung** aufbauen – hier mit einer In-Memory-DB, damit kein Dateimüll entsteht.
2. Einen **Cursor** holen und eine SQL-Anweisung mit `execute(...)` senden.
3. Das Ergebnis mit `fetchall()` (alle Zeilen) oder `fetchone()` (genau eine) abholen.

In [6]:
import sqlite3
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE t (id INTEGER, wert TEXT)")
cur.executemany("INSERT INTO t VALUES (?, ?)", [(1, "a"), (2, "b"), (3, "c")])
conn.commit()


In [7]:
cur.execute("SELECT * FROM t ORDER BY id")
print(cur.fetchall())


[(1, 'a'), (2, 'b'), (3, 'c')]


In [8]:
cur.execute("SELECT wert FROM t WHERE id = ?", (2,))
print(cur.fetchone())
conn.close()


('b',)


## 7. Mini-Übung 1 – Filtern

Finde aus der Liste `wartungen` alle Einträge mit einem `typ` gleich `"Reparatur"` und speichere sie in einer Liste `reparaturen`. Erwartet werden zwei Zeilen.

In [9]:
# TODO: reparaturen = [...]
reparaturen = []

# ▶️ Selbstkontrolle
try:
    assert len(reparaturen) == 2, f"Erwartet 2 Reparaturen, bekommen: {len(reparaturen)}"
    assert all(r["typ"] == "Reparatur" for r in reparaturen), "Nicht alle sind Reparaturen"
    print("✅ Mini-Übung 1 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Variable `reparaturen` fehlt.")


❌ Noch nicht ganz: Erwartet 2 Reparaturen, bekommen: 0


## 8. Mini-Übung 2 – Sortieren mit `key`

Sortiere `wartungen` **aufsteigend** nach `"kosten"` und speichere die sortierte Liste in `sortiert`. Der erste Eintrag soll die günstigste Wartung sein.

In [10]:
# TODO: sortiert = sorted(...)
sortiert = []

# ▶️ Selbstkontrolle
try:
    assert len(sortiert) == 4, "Es müssen 4 Wartungen sein."
    assert sortiert[0]["kosten"] == 90.0, f"Günstigste sollte 90.0 sein, bekommen: {sortiert[0]['kosten']}"
    assert sortiert[-1]["kosten"] == 2500.0, "Teuerste sollte am Ende stehen."
    print("✅ Mini-Übung 2 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Variable `sortiert` fehlt.")


❌ Noch nicht ganz: Es müssen 4 Wartungen sein.


## ✅ Zusammenfassung
- `dict` und `list[dict]` sind die bequemste Python-Darstellung für Tabellen im Speicher.
- `sorted(..., key=lambda x: ...)` sortiert nach beliebigen Feldern, `max`/`min` mit `key` liefern Einzel-Extrema.
- List-Comprehensions mit `if` ersetzen `for`-Schleifen zum Filtern.
- `sqlite3.connect(":memory:")`, `cursor.execute(...)`, `fetchall()` bleiben unser Handwerkszeug aus V19.

## ➡️ Nächster Schritt
Weiter mit [01_theorie.ipynb](01_theorie.ipynb).
